# 08Z · Does an expanded signature recover CD4 or CD8 lineage?

**Adagrasib mouse Visium HD · Held-out validation and annotation**  
Run after 08Y, from the extracted package. The notebook reopens the same count source and frozen signatures. It reports each cell's direct RNA evidence, inferred lineage when supported, pan-T evidence, and independent functional-state scores. A fresh kernel releases discovery-stage memory.

## Select the completed discovery run

In [ ]:
from pathlib import Path
import os
import json
import pandas as pd
from IPython.display import display, Image
from input08y import open_bundle
from workflow08yz import find_discovery, start_scoring, make_review_zip

OUTPUT_ROOT = Path(os.getenv("CT26_08YZ_OUTPUT", "08YZ_results")).expanduser().resolve()
DISCOVERY_DIR = None  # Set the exact completed 08Y directory to revisit an older run.
VALIDATION = dict(target_error=0.02, min_hits=2, min_panT=1, min_balanced_precision=0.80,
                  min_anchors_per_class=20, min_discovery_mice=3, min_evaluation_mice=2,
                  min_correct=15, allow_unknown_mixed=False, max_rank=None, chunk_size=1024)
ydir, receipt = find_discovery(OUTPUT_ROOT, DISCOVERY_DIR)
print("Frozen 08Y run:", ydir)
print("Panels:", json.loads((ydir / "signatures.json").read_text()))

## Score and test each lineage independently

State scores use all measured genes for within-cell ranking. For lineage prediction, all three coreceptors are removed **before ranking**. Strong RNA anchors have one observed coreceptor lineage, detected pan-T evidence, and a negative 08Q mixed flag. Cells with both coreceptors retain that observed category.

Thresholds are learned in the discovery mice. The default target for opposite-lineage calls is 2% of eligible opposite-lineage RNA anchors, with a minimum correct-call requirement. The fixed thresholds are assessed in the evaluation mice. A lineage receives inferred labels only after it passes. Unknown mixed status excludes a cell from the default lineage validation and assignment rules. All cells still receive direct-observation fields and available state scores. The per-gene input ranks use average ties, with no spatial smoothing.

In [ ]:
data = open_bundle(**receipt["source_args"])
zdir = start_scoring(OUTPUT_ROOT, data, ydir, receipt, VALIDATION)
summary = json.loads((zdir / "08Z_complete.json").read_text())
print("Completed scoring:", zdir)
display(pd.DataFrame([{"lineage": lin, "frozen_threshold": summary["thresholds"][lin],
                       "evaluation_passed": summary["passed_evaluation"][lin],
                       "reason": summary["reason"][lin], "inferred_cells": summary["n_inferred"][lin]}
                      for lin in ["CD4", "CD8"]]))

## Check recovery, wrong-lineage calls, and depth coverage

Look at each evaluation mouse. `recovery` is the fraction of eligible target anchors correctly called. `opposite_call_rate` is the fraction of eligible opposite anchors called as the target. `supported_depth_fraction` gives the share of eligible coreceptor-undetected cells represented in depth strata with enough direct anchors for both lineages. Low coverage limits extrapolation to that mouse's unresolved cells.

In [ ]:
validation = pd.read_csv(zdir / "anchor_validation_by_mouse_depth.csv", dtype={"depth_bin": "str"})
display(validation.loc[(validation.split == "evaluation") & (validation.depth_bin == "all"),
                       ["sample", "condition", "lineage", "n_target", "n_opposite", "correct_calls",
                        "opposite_calls", "recovery", "opposite_call_rate", "auc_margin", "passed_evaluation"]])
depth_check = pd.read_csv(zdir / "depth_matched_validation.csv")
display(depth_check.loc[depth_check.split == "evaluation"])
benchmarks = pd.read_csv(zdir / "comparator_auc_by_mouse.csv")
display(benchmarks.loc[benchmarks.split == "evaluation"]
        .pivot(index=["sample", "lineage"], columns="model", values="auc").round(3))
display(Image(filename=str(zdir / "heldout_anchor_performance.png"), width=850))

## Inspect annotation coverage and cell states

The per-cell table preserves observed CD4/CD8 counts, both/one/neither detection, CD4/CD8 score margins, detected members per signature, QC eligibility, and the deployed label. Use `observed_coreceptors` alongside `deployed_lineage` to distinguish direct evidence from a supported prediction. Continuous state scores and observed-member counts are available for lineage-unresolved cells.

In [ ]:
coverage = pd.read_csv(zdir / "lineage_coverage_by_mouse.csv")
display(coverage.pivot_table(index=["sample", "condition"], columns="deployed_lineage",
                             values="cells", aggfunc="sum", fill_value=0))
states = pd.read_csv(zdir / "state_summary_by_mouse.csv")
display(states.query("population == 'coreceptor_neither' and program == 'cytotoxic'"))
display(Image(filename=str(zdir / "lineage_coverage_by_mouse.png"), width=900))
print("Per-cell table:", zdir / "per_cell_annotations.csv.gz")

## Collect the aggregate review files

The review ZIP contains summary tables, settings and figures. Cell IDs, per-cell annotations and expression matrices stay in the run directory. Each result directory is keyed to its inputs, settings, and code. Re-running the same settings reuses a completed result.

In [ ]:
review_files = {
    "08Y": ["observed_coreceptors_by_mouse.csv", "discovery_mouse_counts.csv",
            "reference_detection_by_mouse.csv", "discovery_settings.json", "signatures.json",
            "gene_map.png", "gene_map_evidence.png", "gene_module_summary.csv", "anchor_neighbors_top50.csv",
            "lineage_candidate_audit.csv.gz", "gene_map.csv.gz"],
    "08Z": ["08Z_complete.json", "frozen_thresholds.json", "panel_coverage.csv",
            "anchor_validation_by_mouse_depth.csv", "depth_matched_validation.csv",
            "comparator_auc_by_mouse.csv", "lineage_coverage_by_mouse.csv",
            "state_summary_by_mouse.csv", "heldout_anchor_performance.png",
            "lineage_coverage_by_mouse.png"]
}
print("Review ZIP:", make_review_zip(ydir, zdir, review_files))

## Download the return-to-chat ZIP

This cell can run on its own after a completed analysis. If you used a custom output directory, set `REVIEW_ROOT` below. The ZIP appears beside this notebook and includes an HTML figure index and aggregate results.

In [ ]:
from pathlib import Path
import os
from IPython.display import FileLink, display
from review08yz import return_zip

REVIEW_ROOT = globals().get("OUTPUT_ROOT", Path(os.getenv("CT26_08YZ_OUTPUT", "08YZ_results")))
archive = return_zip(REVIEW_ROOT, stage="08Z", zdir=globals().get("zdir"))
print("Return ZIP:", archive)
display(FileLink(archive.name))